# 🧠 Building a Mini LLM from Scratch

A complete, step-by-step walkthrough of how a **GPT-style language model** is built, from raw text through to generated code. At about **0.9 M parameters** it is tiny next to production models, but every key piece is here, and each piece is shown working before it is used.

The model learns to write **Python code** by reading the Python standard library.

### 🗺️ The pipeline

| # | Stage | What happens |
|---|---|---|
| 1 | Size the model | Pick the hyperparameters and count the parameters |
| 2 | Collect data | Gather the stdlib `.py` files |
| 3 | Clean data | Filter, normalise and de-duplicate them into one corpus |
| 4 | Tokenizer | Train a byte-level BPE vocabulary of 512 tokens |
| 5 | Encode | Turn the corpus into token IDs and split train / held-out |
| 6 | Batches | Build *(input, target)* pairs for next-token prediction |
| 7 | Embedding | Turn IDs into vectors |
| 8 | Attention ("Mix") | Let each position gather information from earlier positions |
| 9 | Feed-forward ("Refine") | Process each position on its own |
| 10 | Full model | Stack the layers into a `MiniLLM` module |
| 11 | Learning step | Loss → backprop → one AdamW update |
| 12 | Training | 2,000 steps with warm-up + cosine learning rate |
| 13 | Generation | Sample new code from the trained model |

### 📖 Vocabulary used in this notebook
- **Table**: a learned weight matrix (one `nn.Parameter`).
- **Grid**: the activation tensor of shape `(batch, position, width)` that flows through the model.
- **Fields**: the `width` numbers that represent one token at one position.
- **Mix**: the attention sub-layer. **Refine**: the feed-forward sub-layer.

> **Requirements:** Python 3.14 + PyTorch. Runs on the Apple-silicon GPU (`mps`) or on the CPU.
> Each code cell is followed by its recorded output, so the notebook can be read without running it.

---
## 1️⃣ Sizing the model

Before writing any model code, we fix the hyperparameters and count how many learnable numbers (parameters) the model will have. Each transformer layer holds:

- **4 attention tables** (`Q`, `K`, `V`, `O`), each `W × W`
- **2 feed-forward tables** (`Up`: `W × I`, `Down`: `I × W`)
- **2 RMSNorm scale vectors** of length `W`

The **embedding** table (`V × W`), a **final norm** (`W`) and an **output** table (`W × V`) sit around the stack of layers:

$$\text{params} = \underbrace{VW}_{\text{embedding}} + L\,(4W^2 + 2WI + 2W) + \underbrace{W}_{\text{final norm}} + \underbrace{WV}_{\text{output}}$$

In [ ]:
# ── Model configuration ──────────────────────────────────────────────────────
#   vocab  : number of distinct token IDs the model knows
#   width  : number of fields (vector size) used to represent each token
#   layers : number of transformer blocks stacked on top of each other
#   heads  : attention heads per block (the width is split evenly between them)
#   inner  : hidden size of the feed-forward "Refine" step (4 x width)
#   seq    : context length - how many tokens the model reads at once
#   batch  : how many sequences are processed in parallel per training step
cfg = dict(vocab=512, width=128, layers=4, heads=4, inner=512, seq=128, batch=32)

V, W, L, I = cfg["vocab"], cfg["width"], cfg["layers"], cfg["inner"]

# Size of every weight table inside ONE transformer layer
per_layer = {"Q": W*W, "K": W*W, "V": W*W, "O": W*W,   # attention projections
             "Up": W*I, "Down": I*W,                   # feed-forward expand / shrink
             "norm1": W, "norm2": W}                   # RMSNorm scale vectors
total = V*W + L*sum(per_layer.values()) + W + V*W   # Embedding + layers + final norm + Output

# table count = 1 embedding + 8 per layer + 1 final norm + 1 output
print("tables:", 1 + L*len(per_layer) + 1 + 1)
print("total params:", f"{total:,}")

**Output**

```code
tables: 35
total params: 918,656
```

> 💡 **What to notice:** These numbers are checked again in section 10, where the real PyTorch model reports the same counts.

---
## 2️⃣ Collecting the training data

Any language model needs a large body of text to learn from. Here that text is the **Python standard library** that ships with the interpreter: roughly 10 MB of well-written, consistently styled code, already on disk, so nothing needs to be downloaded.

Test suites, IDLE (the IDE), third-party packages and demo folders are skipped because they would skew the style of the data.

In [ ]:
import sysconfig, pathlib

# Locate the standard-library folder of the Python interpreter running this notebook
root = pathlib.Path(sysconfig.get_paths()["stdlib"])

# Folders that are not representative library code (tests, IDE, 3rd-party, demos)
skip = {"test", "tests", "idlelib", "site-packages", "lib2to3", "turtledemo"}

# Every .py file whose path does not pass through one of the skipped folders
files = [p for p in root.rglob("*.py") if not skip & set(p.parts)]

print(root)
print("files:", len(files))
print("raw bytes:", f"{sum(p.stat().st_size for p in files):,}")

**Output**

```code
/opt/homebrew/opt/python@3.14/Frameworks/Python.framework/Versions/3.14/lib/python3.14
files: 575
raw bytes: 10,887,689
```

---
## 3️⃣ Cleaning the corpus

Raw data is rarely ready for training. Real LLM data pipelines run filtering, normalisation and de-duplication, and this cell runs a small version of each:

1. **Decode**: keep only files that are valid UTF-8.
2. **Drop near-duplicates**: the `encodings/` package holds ~120 machine-generated codec files that look almost the same.
3. **Size filter**: very large files are mostly generated data tables; very small ones are mostly empty `__init__.py` files.
4. **Normalise**: strip the leading comment block (shebang / licence), turn tabs into 4 spaces, remove trailing spaces and collapse long runs of blank lines.
5. **Exact de-duplication**: hash every file and keep only the first copy of each.
6. **Join**: join everything into one `corpus.txt`, with an `<|eof|>` marker between files so the model can learn where a file ends.

The `stats` table tracks how many files and bytes survive each step.

In [ ]:
import hashlib, re

# ── Bookkeeping: record how many files / bytes survive each cleaning step ────
def mb(n): return f"{n/1e6:.2f} MB"                                       # bytes -> "x.xx MB"
stats = {"start": [len(files), sum(p.stat().st_size for p in files)]}    # step -> [files, bytes]
def note(name, kept): stats[name] = [len(kept), sum(len(t.encode()) for _, t in kept)]

# 1. read; skip anything that is not valid UTF-8
docs = []                                        # list of (path, text) pairs
for p in files:
    try: docs.append((p, p.read_text(encoding="utf-8")))
    except UnicodeDecodeError: pass
note("utf-8 readable", docs)

# 2. near-duplicates: encodings/ is ~120 generated codec files that look alike
docs = [(p, t) for p, t in docs if "encodings" not in p.parts]
note("drop near-duplicates", docs)

# 3. too big = generated data tables; too small = empty __init__ files
docs = [(p, t) for p, t in docs if 200 < len(t) < 150_000]
note("size filter", docs)

# 4. normalise: drop leading # block (shebang, licence), tabs -> 4 spaces,
#    no trailing spaces, at most 2 blank lines in a row
def normalise(t):
    lines = t.split("\n")
    while lines and (lines[0].startswith("#") or not lines[0].strip()):   # strip header comments / blank lines
        lines.pop(0)
    t = "\n".join(l.replace("\t", "    ").rstrip() for l in lines)       # tabs -> spaces, trim line ends
    return re.sub(r"\n{4,}", "\n\n\n", t).strip() + "\n"                 # cap blank-line runs at 2
docs = [(p, normalise(t)) for p, t in docs]
note("normalise", docs)

# 5. exact duplicates (same content after normalising)
seen, unique = set(), []
for p, t in docs:
    h = hashlib.sha1(t.encode()).hexdigest()     # content fingerprint
    if h not in seen:                            # keep only the first file with this fingerprint
        seen.add(h); unique.append((p, t))
docs = unique
note("exact dedup", docs)

# 6. join into one corpus; files separated by an end-of-file marker
EOF_MARK = "<|eof|>"
corpus = f"\n{EOF_MARK}\n".join(t for _, t in docs)
pathlib.Path("corpus.txt").write_text(corpus, encoding="utf-8")

# ── Report: files and size remaining after every step, plus a peek at the corpus
for step, (n, b) in stats.items():
    print(f"{step:22} {n:5} files  {mb(b)}")
print("corpus chars:", f"{len(corpus):,}")
print(corpus[:300])

**Output**

```code
start                    575 files  10.89 MB
utf-8 readable           575 files  10.89 MB
drop near-duplicates     452 files  9.47 MB
size filter              424 files  8.46 MB
normalise                424 files  8.41 MB
exact dedup              423 files  8.41 MB
corpus chars: 8,416,057
"""Utility functions for copying and archiving files and directory trees.

XXX The functions here don't copy the resource fork or other metadata on Mac.

"""

import os
import sys
import stat
import fnmatch
import collections
import errno

try:
    import zlib
    del zlib
    _ZLIB_SUPPORTED = True
```

> 💡 **What to notice:** About 22% of the raw bytes are removed, mostly by dropping the generated `encodings/` codec files and oversized files.

---
## 4️⃣ Training a BPE tokenizer

Models work with integer **token IDs**, not characters. **Byte-Pair Encoding (BPE)** builds a vocabulary from the data itself:

1. **Pre-split** the text into chunks (words, numbers, punctuation, whitespace) so merges never cross chunk boundaries. For example, `" self"` can become a token but `"self."` cannot.
2. **Start from bytes**: every chunk becomes a list of UTF-8 byte values (IDs 0–255), so *any* text can be encoded and no character is ever unknown.
3. **Merge repeatedly**: count every adjacent pair of IDs (weighted by how often its chunk appears), turn the most frequent pair into a new ID, and repeat.

After **255 merges** plus one special `<|eof|>` ID, the vocabulary has exactly **512 tokens** (`256 bytes + 255 merges + 1`).

In [ ]:
import collections, time

# 1. split into chunks: a word with its leading space, a number, punctuation, a run of spaces
#    Regex alternatives, tried left to right:
#      " ?[A-Za-z_]+"        identifier / word (with optional leading space)
#      " ?\d+"               number
#      " ?[^\sA-Za-z_\d]+"   run of punctuation / operators
#      "\s+(?!\S)"           whitespace, leaving the last space to attach to the next word
#      "\s+"                 any remaining whitespace
SPLIT = re.compile(r" ?[A-Za-z_]+| ?\d+| ?[^\sA-Za-z_\d]+|\s+(?!\S)|\s+")
text = corpus.replace(EOF_MARK, "")             # the marker gets its own ID, not merges
chunk_freq = collections.Counter(SPLIT.findall(text))   # chunk -> how often it occurs
print("chunks:", f"{sum(chunk_freq.values()):,}", "| unique:", f"{len(chunk_freq):,}")

# 2. every unique chunk as a list of byte IDs (0-255): the starting alphabet
words = {c: list(c.encode("utf-8")) for c in chunk_freq}

# 3. merge loop: count adjacent pairs (weighted by chunk frequency), merge the top one, repeat
NUM_MERGES = 255
merges = {}                                     # (a, b) -> new ID, in the order learned
vocab = {i: bytes([i]) for i in range(256)}     # ID -> the bytes it stands for
t0 = time.time()
for k in range(NUM_MERGES):
    # count every adjacent ID pair; a chunk seen f times contributes f to each of its pairs
    pairs = collections.Counter()
    for c, ids in words.items():
        f = chunk_freq[c]
        for a, b in zip(ids, ids[1:]):
            pairs[(a, b)] += f
    # the most frequent pair becomes a brand-new token
    (a, b), count = pairs.most_common(1)[0]
    new_id = 256 + k
    merges[(a, b)] = new_id
    vocab[new_id] = vocab[a] + vocab[b]
    for c, ids in words.items():                # rewrite every chunk using the new piece
        out, i = [], 0
        while i < len(ids):
            if i < len(ids) - 1 and ids[i] == a and ids[i+1] == b:
                out.append(new_id); i += 2      # pair found -> replace with the merged ID
            else:
                out.append(ids[i]); i += 1      # otherwise copy the ID unchanged
        words[c] = out
    if k < 10 or k % 50 == 0:                   # log the first 10 merges, then every 50th
        print(f"merge {k:3}: {vocab[a]!r} + {vocab[b]!r} -> {new_id} {vocab[new_id]!r}  (seen {count:,}x)")

# reserve the last ID for the end-of-file marker
EOF_ID = 256 + NUM_MERGES                       # 511
vocab[EOF_ID] = EOF_MARK.encode()
print("vocab size:", len(vocab), f"| trained in {time.time()-t0:.0f}s")
print("last 10 pieces:", [vocab[i].decode("utf-8", "replace") for i in range(EOF_ID-10, EOF_ID)])

**Output**

```code
chunks: 1,759,920 | unique: 63,904
merge   0: b' ' + b' ' -> 256 b'  '  (seen 1,387,085x)
merge   1: b'  ' + b'  ' -> 257 b'    '  (seen 511,164x)
merge   2: b'  ' + b' ' -> 258 b'   '  (seen 175,607x)
merge   3: b'\n' + b'    ' -> 259 b'\n    '  (seen 128,225x)
merge   4: b's' + b'e' -> 260 b'se'  (seen 90,579x)
merge   5: b'i' + b'n' -> 261 b'in'  (seen 75,178x)
merge   6: b'r' + b'e' -> 262 b're'  (seen 68,870x)
merge   7: b'    ' + b'    ' -> 263 b'        '  (seen 64,580x)
merge   8: b'\n    ' + b'   ' -> 264 b'\n       '  (seen 64,207x)
merge   9: b' ' + b't' -> 265 b' t'  (seen 59,511x)
merge  50: b' ' + b'e' -> 306 b' e'  (seen 20,126x)
merge 100: b't' + b'h' -> 356 b'th'  (seen 9,812x)
merge 150: b'la' + b'ss' -> 406 b'lass'  (seen 5,810x)
merge 200: b'u' + b'le' -> 456 b'ule'  (seen 4,280x)
merge 250: b'ar' + b'gs' -> 506 b'args'  (seen 3,401x)
vocab size: 512 | trained in 39s
last 10 pieces: ['class', 'mple', 'ize', 'qu', 'ader', 'args', ' al', 'Re', 'el', 'mat']
```

> 💡 **What to notice:** The first merges learned are all **indentation** (runs of spaces), the most common pattern in Python source. Keywords and common identifiers such as `class` and `args` appear later.

---
## 5️⃣ Encoding the corpus

To apply the tokenizer to new text, the learned merges are replayed **in the order they were learned**: within each chunk, the pair that was learned earliest is merged first, until no learned pair is left. A cache saves the result for each chunk, since the same chunks repeat millions of times.

- `encode(str) → list[int]` and `decode(list[int]) → str` are exact inverses (checked with an `assert`).
- Each file is encoded on its own, followed by `EOF_ID`.
- The last **10%** of tokens are held out as a validation set and never trained on, so we can check whether the model *generalises* rather than memorises.

In [ ]:
import torch

rank = {pair: i for i, pair in enumerate(merges)}   # earlier merge = higher priority
cache = {}                                          # chunk -> its token IDs (chunks repeat a lot)

def encode_chunk(c):
    """Turn one pre-split chunk into token IDs by replaying the learned merges."""
    if c in cache: return cache[c]
    ids = list(c.encode("utf-8"))                   # start from raw bytes
    while len(ids) > 1:
        pair = min(zip(ids, ids[1:]), key=lambda p: rank.get(p, 1e9))  # earliest-learned pair
        if pair not in merges: break                # nothing left to merge
        new, out, i = merges[pair], [], 0
        while i < len(ids):                         # replace every occurrence of that pair
            if i < len(ids) - 1 and (ids[i], ids[i+1]) == pair:
                out.append(new); i += 2
            else:
                out.append(ids[i]); i += 1
        ids = out
    cache[c] = ids
    return ids

def encode(s): return [i for c in SPLIT.findall(s) for i in encode_chunk(c)]                  # text -> IDs
def decode(ids): return b"".join(vocab[i] for i in ids).decode("utf-8", errors="replace")    # IDs -> text

# tiny example first
line = "    def __init__(self, name):"
ids = encode(line)
print("IDs:   ", ids)
print("pieces:", [vocab[i].decode() for i in ids])

# whole corpus: encode each file, EOF_ID between files
all_ids = []
for doc in corpus.split(f"\n{EOF_MARK}\n"):
    all_ids += encode(doc) + [EOF_ID]
print("characters:", f"{len(corpus):,}", "| tokens:", f"{len(all_ids):,}",
      f"| {len(corpus)/len(all_ids):.2f} chars per token")
assert decode(all_ids[:2000]).startswith(corpus[:500])   # round trip works

# hold out the last 10%
data = torch.tensor(all_ids, dtype=torch.long)
n = int(0.9 * len(data))
train_ids, val_ids = data[:n], data[n:]
print("train:", train_ids.shape, "| held-out:", val_ids.shape)

**Output**

```code
IDs:    [258, 338, 457, 261, 294, 309, 40, 277, 44, 440, 302]
pieces: ['   ', ' def', ' __', 'in', 'it', '__', '(', 'self', ',', ' name', '):']
characters: 8,416,057 | tokens: 3,906,008 | 2.15 chars per token
train: torch.Size([3515407]) | held-out: torch.Size([390601])
```

> 💡 **What to notice:** `__init__` splits into meaningful pieces (`' __'`, `'in'`, `'it'`, `'__'`). At **2.15 characters per token**, the model sees less than half as many steps as it would reading single characters.

---
## 6️⃣ Batching: next-token prediction

An LLM is trained on a single, simple objective: **predict the next token**. A training example is a random window of `SEQ` tokens:

- `x` = tokens `[s, s+SEQ)`: what the model reads
- `y` = tokens `[s+1, s+SEQ+1)`: the same window **shifted one step to the left**

So every position in the window is its own prediction task: from tokens `0..t`, guess token `t+1`. One window of length 128 gives **128 training signals** at once.

Training runs on the Apple-silicon GPU (`mps`) when available, otherwise on the CPU.

In [ ]:
torch.manual_seed(0)                                              # reproducible batches
device = "mps" if torch.backends.mps.is_available() else "cpu"    # Apple GPU if present
SEQ, BATCH = cfg["seq"], cfg["batch"]

def get_batch(split):
    """Sample BATCH random windows of SEQ tokens and their next-token targets."""
    src = train_ids if split == "train" else val_ids
    starts = torch.randint(0, len(src) - SEQ - 1, (BATCH,))      # random window start positions
    x = torch.stack([src[s : s + SEQ]         for s in starts])   # input
    y = torch.stack([src[s + 1 : s + SEQ + 1] for s in starts])   # target = input shifted by 1
    return x.to(device), y.to(device)

x, y = get_batch("train")
print("device:", device)
print("x (input):", x.shape, "| y (target):", y.shape)

# look at the first 8 positions of sequence 0: each target is the next input
for t in range(8):
    print(f"pos {t}: sees {vocab[x[0,t].item()]!r:14} -> must guess {vocab[y[0,t].item()]!r}")

**Output**

```code
device: mps
x (input): torch.Size([32, 128]) | y (target): torch.Size([32, 128])
pos 0: sees b'get'         -> must guess b'me'
pos 1: sees b'me'          -> must guess b'm'
pos 2: sees b'm'           -> must guess b'b'
pos 3: sees b'b'           -> must guess b'er'
pos 4: sees b'er'          -> must guess b's'
pos 5: sees b's'           -> must guess b'()'
pos 6: sees b'()'          -> must guess b' is'
pos 7: sees b' is'         -> must guess b' c'
```

> 💡 **What to notice:** Every target is simply the *next* input, the whole idea of language-model training.

---
## 7️⃣ Embedding: from IDs to vectors

A token ID is just an index. The **embedding table** (`V × W`) gives each ID a learned vector of `W = 128` fields. Looking up every ID in the batch turns the `(batch, seq)` tensor of IDs into a **grid** of shape `(batch, seq, width)`, and that grid is what flows through the rest of the model.

> ⚠️ From here on the walkthrough uses a **longer context (256) and smaller batch (16)**. These settings also carry through to the training run in section 12.

The cell proves the embedding is nothing more than a table lookup.

In [ ]:
import torch.nn as nn

# Walkthrough settings from here on: longer context, smaller batch (also used for training)
cfg["seq"], cfg["batch"] = 256, 16
SEQ, BATCH = cfg["seq"], cfg["batch"]
x, y = get_batch("train")
print("x (IDs):", x.shape)                   # (batch, seq) of integer token IDs

emb = nn.Embedding(V, W).to(device)          # the Embedding TABLE: one row per vocab ID
print("Embedding table:", emb.weight.shape)

grid = emb(x)                                # lookup: every ID -> its row
print("grid:", grid.shape)                   # (batch, seq, width)

# proof it's just a lookup: position 0 of sequence 0 is exactly one row of the table
first_id = x[0, 0].item()
print("ID at [0,0]:", first_id, repr(vocab[first_id]))
print("grid[0,0] == table row", first_id, "->", torch.equal(grid[0, 0], emb.weight[first_id]))
print("first 6 fields:", grid[0, 0, :6].tolist())

**Output**

```code
x (IDs): torch.Size([16, 256])
Embedding table: torch.Size([512, 128])
grid: torch.Size([16, 256, 128])
ID at [0,0]: 98 b'b'
grid[0,0] == table row 98 -> True
first 6 fields: [-1.0012174844741821, 0.4069393277168274, -0.2992555797100067, 0.9639763236045837, 0.37417203187942505, -1.8978575468063354]
```

> 💡 **What to notice:** `grid[0,0]` equals the row of the embedding table for that ID exactly: an embedding is a lookup, not a computation.

---
## 8️⃣ Attention: the "Mix" step

Attention is the only place where positions **share information**. The steps:

1. **RMSNorm** keeps the scale of every row steady before it goes into the projections.
2. **Project** every row with three tables into a **query** (*what am I looking for?*), a **key** (*what do I contain?*) and a **value** (*what will I pass on?*).
3. **Split into heads**: the 128 fields become 4 heads × 32 fields, so each head can learn its own kind of relationship.
4. **RoPE** (rotary position embedding) rotates queries and keys by an angle that depends on their position, so a query·key score depends on the *relative* distance between two tokens.
5. **Scores and causal mask**: each query is compared with every key. Future positions are masked to $-\infty$ so the model cannot cheat by looking ahead, then softmax turns each row into weights that sum to 1.
6. **Combine**: the weighted values are summed, the heads are joined back together, the `O` table projects the result, and it is **added** to the grid (a *residual connection*).

$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d}} + \text{mask}\right)V$$

In [ ]:
import math
import torch.nn.functional as F
H = cfg["heads"]; D = W // H                       # 4 heads x 32 fields

norm1 = nn.RMSNorm(W).to(device)
Wq, Wk, Wv, Wo = [nn.Linear(W, W, bias=False).to(device) for _ in range(4)]   # the 4 TABLES
print("Q table:", Wq.weight.shape)

# 1. pre-norm: rescale every row so its values have a steady size
h = norm1(grid)

# the shape rule on ONE row: (128) @ (128 x 128) -> (128)
row = h[0, 0]
print("one row", row.shape, "@ Q table", Wq.weight.T.shape, "->", (row @ Wq.weight.T).shape)

# 2. whole grid through Q, K, V
q, k, v = Wq(h), Wk(h), Wv(h)                      # query / key / value for every position
print("q, k, v:", q.shape)

# 3. split 128 fields into 4 heads x 32
def split(t): return t.view(BATCH, SEQ, H, D).transpose(1, 2)
q, k, v = split(q), split(k), split(v)
print("per head:", q.shape, " (batch, head, position, field)")

# 4. RoPE: rotate q and k by position
def rope(t):
    half = D // 2
    freqs = 1.0 / (10000 ** (torch.arange(half, device=device) / half))   # one rotation speed per field pair
    ang = torch.arange(SEQ, device=device)[:, None] * freqs[None, :]   # (256, 16)
    t1, t2 = t[..., :half], t[..., half:]
    # 2-D rotation of each (t1, t2) pair by its angle
    return torch.cat([t1 * ang.cos() - t2 * ang.sin(), t1 * ang.sin() + t2 * ang.cos()], dim=-1)
q, k = rope(q), rope(k)

# 5. scores: every position vs every position, future masked
scores = q @ k.transpose(-2, -1) / math.sqrt(D)    # scaled dot products
print("scores:", scores.shape)
mask = torch.tril(torch.ones(SEQ, SEQ, device=device, dtype=torch.bool))     # True = allowed (past + self)
weights = F.softmax(scores.masked_fill(~mask, float("-inf")), dim=-1)        # each row sums to 1
print("weights, head 0, first 4 positions:\n", weights[0, 0, :4, :4].detach().cpu().numpy().round(2))
mixed = weights @ v                                # weighted sum of values
print("mixed per head:", mixed.shape)

# 6. glue heads back, O table, residual add
mixed = mixed.transpose(1, 2).reshape(BATCH, SEQ, W)
print("heads glued:", mixed.shape)
grid = grid + Wo(mixed)                            # residual: add, don't overwrite
print("grid after Mix:", grid.shape)

**Output**

```code
Q table: torch.Size([128, 128])
one row torch.Size([128]) @ Q table torch.Size([128, 128]) -> torch.Size([128])
q, k, v: torch.Size([16, 256, 128])
per head: torch.Size([16, 4, 256, 32])  (batch, head, position, field)
scores: torch.Size([16, 4, 256, 256])
weights, head 0, first 4 positions:
 [[1.   0.   0.   0.  ]
 [0.74 0.26 0.   0.  ]
 [0.28 0.34 0.38 0.  ]
 [0.22 0.33 0.23 0.21]]
mixed per head: torch.Size([16, 4, 256, 32])
heads glued: torch.Size([16, 256, 128])
grid after Mix: torch.Size([16, 256, 128])
```

> 💡 **What to notice:** The attention weights are **lower-triangular**: position 0 can only see itself, while position 3 spreads its attention over positions 0–3. Each row sums to 1. The grid's shape is unchanged after Mix.

---
## 9️⃣ Feed-forward: the "Refine" step

After attention has mixed information across positions, the feed-forward network processes **each position on its own**:

1. **RMSNorm** the grid.
2. **Up**: expand each row from 128 to 512 fields (`inner = 4 × width`).
3. **GELU**: a smooth non-linearity that pushes negative values towards zero. Without it, stacking linear layers would add up to a single linear map.
4. **Down**: shrink back to 128 fields.
5. **Residual add**: add the result to the grid.

One **Mix + Refine** pair is a single transformer layer. The grid keeps its shape `(16, 256, 128)`, which is why the layers can be stacked.

In [ ]:
norm2 = nn.RMSNorm(W).to(device)
Up   = nn.Linear(W, I, bias=False).to(device)    # TABLE (128 -> 512)
Down = nn.Linear(I, W, bias=False).to(device)    # TABLE (512 -> 128)
print("Up table:", Up.weight.T.shape, "| Down table:", Down.weight.T.shape)

# 1. pre-norm
h = norm2(grid)

# 2. expand each row 128 -> 512 fields
wide = Up(h)
print("after Up:", wide.shape)

# 3. non-linearity: GELU softly clips negative values towards zero
act = F.gelu(wide)
print("fields near zero after clip, row [0,0]:", (act[0, 0].abs() < 0.05).sum().item(), "of", I)

# 4. shrink back 512 -> 128 fields
back = Down(act)
print("after Down:", back.shape)

# 5. residual add: one full transformer layer (Mix + Refine) is done
grid = grid + back
print("grid after layer 1:", grid.shape)

**Output**

```code
Up table: torch.Size([128, 512]) | Down table: torch.Size([512, 128])
after Up: torch.Size([16, 256, 512])
fields near zero after clip, row [0,0]: 72 of 512
after Down: torch.Size([16, 256, 128])
grid after layer 1: torch.Size([16, 256, 128])
```

> 💡 **What to notice:** In the untrained network, GELU pushes 72 of 512 fields close to zero. The grid comes back to its original shape, ready for the next layer.

---
## 🔟 The complete model

Now everything from sections 7–9 is packed into reusable modules:

- **`Block`**: one transformer layer (Mix + Refine, each with a pre-norm and a residual). It uses PyTorch's fused `scaled_dot_product_attention`, which does the same scores → mask → softmax → weighted sum as section 8, only faster.
- **`MiniLLM`**: embedding → 4 blocks → final RMSNorm → output table, which gives a **logit** (raw score) for each of the 512 vocabulary entries at every position.

`rope` is redefined to take the sequence length `T` as an argument, so the model can handle prompts of any length up to `SEQ` during generation.

In [ ]:
# RoPE again, now taking the sequence length T so any prompt length <= SEQ works
def rope(t, T):
    half = D // 2
    freqs = 1.0 / (10000 ** (torch.arange(half, device=t.device) / half))
    ang = torch.arange(T, device=t.device)[:, None] * freqs[None, :]
    t1, t2 = t[..., :half], t[..., half:]
    return torch.cat([t1 * ang.cos() - t2 * ang.sin(), t1 * ang.sin() + t2 * ang.cos()], dim=-1)

class Block(nn.Module):                          # one layer: Mix + Refine
    def __init__(self):
        super().__init__()
        self.norm1, self.norm2 = nn.RMSNorm(W), nn.RMSNorm(W)
        self.Wq, self.Wk, self.Wv, self.Wo = [nn.Linear(W, W, bias=False) for _ in range(4)]   # attention tables
        self.Up, self.Down = nn.Linear(W, I, bias=False), nn.Linear(I, W, bias=False)          # feed-forward tables

    def forward(self, g):
        B, T, _ = g.shape
        h = self.norm1(g)
        sp = lambda t: t.view(B, T, H, D).transpose(1, 2)             # split into heads
        q, k, v = sp(self.Wq(h)), sp(self.Wk(h)), sp(self.Wv(h))
        q, k = rope(q, T), rope(k, T)
        m = F.scaled_dot_product_attention(q, k, v, is_causal=True)   # fused scores+mask+softmax+sum
        g = g + self.Wo(m.transpose(1, 2).reshape(B, T, W))           # Mix + residual
        g = g + self.Down(F.gelu(self.Up(self.norm2(g))))             # Refine + residual
        return g

class MiniLLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(V, W)                                  # Embedding TABLE
        self.blocks = nn.ModuleList([Block() for _ in range(L)])       # L stacked layers
        self.norm = nn.RMSNorm(W)                                      # final norm
        self.out = nn.Linear(W, V, bias=False)                         # Output TABLE

    def forward(self, ids, show=False):
        g = self.emb(ids)                                              # IDs -> grid
        if show: print("grid after embedding:", g.shape)
        for i, b in enumerate(self.blocks):
            g = b(g)
            if show: print(f"grid after layer {i+1}:", g.shape)
        logits = self.out(self.norm(g))                                # grid -> score per vocab ID
        if show: print("scores per vocab ID:", logits.shape)
        return logits

# Build the model and confirm it matches the hand count from section 1
torch.manual_seed(0)
model = MiniLLM().to(device)
print("tables:", len(list(model.parameters())),
      "| params:", f"{sum(p.numel() for p in model.parameters()):,}")

# One forward pass, printing the grid shape at every stage
logits = model(x, show=True)
probs = F.softmax(logits, dim=-1)                # logits -> probabilities
print("probs:", probs.shape, "| sum over vocab at [0,0]:", round(probs[0, 0].sum().item(), 4))

# What does the untrained model predict after the last position?
top = probs[0, -1].topk(5)
print("untrained top-5 guesses after the last position:")
for p, i in zip(top.values.tolist(), top.indices.tolist()):
    print(f"  {vocab[i]!r:12} {p:.4f}")
print("uniform guess would be:", round(1 / V, 4))

**Output**

```code
tables: 35 | params: 918,656
grid after embedding: torch.Size([16, 256, 128])
grid after layer 1: torch.Size([16, 256, 128])
grid after layer 2: torch.Size([16, 256, 128])
grid after layer 3: torch.Size([16, 256, 128])
grid after layer 4: torch.Size([16, 256, 128])
scores per vocab ID: torch.Size([16, 256, 512])
probs: torch.Size([16, 256, 512]) | sum over vocab at [0,0]: 1.0
untrained top-5 guesses after the last position:
  b'X'         0.0139
  b'get'       0.0075
  b'\xed'      0.0073
  b'2'         0.0073
  b'mple'      0.0061
uniform guess would be: 0.002
```

> 💡 **What to notice:** **35 tables and 918,656 parameters**, an exact match with section 1. The untrained model's probabilities are close to uniform (1/512 ≈ 0.002): it has not learned anything yet.

---
## 1️⃣1️⃣ How the model learns: one step

Training repeats three operations:

1. **Loss**: the **cross-entropy** between the predicted distribution and the real next token, which is $-\log p(\text{correct token})$ averaged over all positions. A model that guesses uniformly scores $\ln 512 \approx 6.24$.
2. **Backpropagation**: `loss.backward()` computes a **gradient** for every parameter, a table with the same shape that says which direction would lower the loss.
3. **Update**: the **AdamW** optimizer nudges every table a small step against its gradient (with weight decay for regularisation).

The cell runs a single step and checks that the loss on the same batch goes down.

In [ ]:
# 1. loss
loss = F.cross_entropy(logits.view(-1, V), y.view(-1))     # average -log p(correct) over all positions
print("loss:", round(loss.item(), 3), "| ln(512) =", round(math.log(V), 3))

# the same thing by hand for ONE position
p = probs[0, 0, y[0, 0]].item()
print(f"position [0,0]: right answer {vocab[y[0,0].item()]!r}, given p={p:.5f}, -log(p)={-math.log(p):.3f}")

# 2. backprop: a gradient table next to every table, same shape
model.zero_grad()
loss.backward()
Wq1 = model.blocks[0].Wq.weight
print("layer-1 Q table:", Wq1.shape, "| its gradient:", Wq1.grad.shape)

# 3. one AdamW update
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.1)
before = Wq1[0, :5].detach().clone()
opt.step()                                                  # every table moves a small step against its gradient
after = Wq1[0, :5].detach().clone()
print("Q[0,:5] before:", before.cpu().numpy().round(4))
print("Q[0,:5] after: ", after.cpu().numpy().round(4))
print("change:        ", (after - before).cpu().numpy().round(4))

# 4. same batch again: a bit less wrong?
with torch.no_grad():
    loss2 = F.cross_entropy(model(x).view(-1, V), y.view(-1))
print("loss before:", round(loss.item(), 4), "-> after one step:", round(loss2.item(), 4))

**Output**

```code
loss: 6.422 | ln(512) = 6.238
position [0,0]: right answer b'.', given p=0.00100, -log(p)=6.903
layer-1 Q table: torch.Size([128, 128]) | its gradient: torch.Size([128, 128])
Q[0,:5] before: [-0.0863 -0.0703 -0.0501 -0.0639  0.0282]
Q[0,:5] after:  [-0.0853 -0.0693 -0.0511 -0.0629  0.0291]
change:         [ 0.001  0.001 -0.001  0.001  0.001]
loss before: 6.4223 -> after one step: 6.139
```

> 💡 **What to notice:** The starting loss (6.42) is close to the uniform-guess value ln(512) = 6.24, and a single update already lowers it to 6.14.

---
## 1️⃣2️⃣ Training

The same step, repeated 2,000 times on fresh random batches, with a few standard extras:

- **Learning-rate schedule**: a linear **warm-up** over 100 steps (to avoid unstable early updates), then **cosine decay** to 10% of the peak.
- **Gradient clipping** limits the size of the gradient to 1.0, so one odd batch cannot throw training off.
- Every 200 steps, the loss is **evaluated** on both the training and the held-out data (20 batches each, no gradients).

When training finishes, the weights, config and tokenizer (`merges`, `vocab`) are saved to `mini_llm_v1.pt`.

In [ ]:
import time
torch.manual_seed(1)
model = MiniLLM().to(device)                      # fresh, untrained model
STEPS, WARMUP, PEAK_LR = 2000, 100, 2e-3
opt = torch.optim.AdamW(model.parameters(), lr=PEAK_LR, weight_decay=0.1)

def lr_at(step):                                  # warm-up, then cosine decay to 10%
    if step < WARMUP: return PEAK_LR * (step + 1) / WARMUP
    progress = (step - WARMUP) / (STEPS - WARMUP)                     # 0 -> 1 after warm-up
    return PEAK_LR * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * progress)))

@torch.no_grad()
def eval_loss(split, n=20):
    """Average loss over n random batches, without updating the model."""
    model.eval()
    tot = sum(F.cross_entropy(model(xb).view(-1, V), yb.view(-1)).item()
              for xb, yb in (get_batch(split) for _ in range(n)))
    model.train()
    return tot / n

t0 = time.time()
for step in range(STEPS + 1):
    # ── progress report every 200 steps ──
    if step % 200 == 0:
        tr, va, el = eval_loss("train"), eval_loss("val"), time.time() - t0
        tok_s = step * BATCH * SEQ / el if step else 0                # training throughput
        print(f"step {step:5} | lr {lr_at(step):.5f} | train {tr:.3f} | held-out {va:.3f} | {tok_s:,.0f} tok/s | {el:.0f}s")
    if step == STEPS: break
    # ── one training step ──
    for g in opt.param_groups: g["lr"] = lr_at(step)                 # apply the schedule
    xb, yb = get_batch("train")
    loss = F.cross_entropy(model(xb).view(-1, V), yb.view(-1))   # read + judge
    opt.zero_grad(set_to_none=True)
    loss.backward()                                              # backprop
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)      # cap any wild step
    opt.step()                                                   # write

# Save weights + config + tokenizer so the model can be reloaded later
torch.save({"model": model.state_dict(), "cfg": cfg, "merges": merges, "vocab": vocab}, "mini_llm_v1.pt")
print("saved mini_llm_v1.pt")

**Output**

```code
step     0 | lr 0.00002 | train 6.385 | held-out 6.391 | 0 tok/s | 0s
step   200 | lr 0.00199 | train 3.442 | held-out 3.459 | 112,824 tok/s | 7s
step   400 | lr 0.00189 | train 2.866 | held-out 2.984 | 116,120 tok/s | 14s
step   600 | lr 0.00171 | train 2.572 | held-out 2.700 | 115,588 tok/s | 21s
step   800 | lr 0.00146 | train 2.452 | held-out 2.578 | 116,566 tok/s | 28s
step  1000 | lr 0.00117 | train 2.359 | held-out 2.551 | 117,030 tok/s | 35s
step  1200 | lr 0.00088 | train 2.315 | held-out 2.472 | 117,672 tok/s | 42s
step  1400 | lr 0.00061 | train 2.200 | held-out 2.437 | 118,117 tok/s | 49s
step  1600 | lr 0.00039 | train 2.115 | held-out 2.374 | 118,486 tok/s | 55s
step  1800 | lr 0.00025 | train 2.130 | held-out 2.352 | 118,354 tok/s | 62s
step  2000 | lr 0.00020 | train 2.095 | held-out 2.337 | 117,928 tok/s | 69s
saved mini_llm_v1.pt
```

> 💡 **What to notice:** Held-out loss falls from **6.39 → 2.34** in about 70 seconds (~2.3 passes over the training data). The gap between train and held-out loss grows slowly, an early sign of overfitting.

---
## 1️⃣3️⃣ Generating code

Generation is **autoregressive**: predict a distribution for the next token, sample one, append it, repeat.

- Only the **last `SEQ` tokens** are fed back in (the model's context window).
- **Temperature** (`0.8`) sharpens the distribution: values below 1 make choices more conservative.
- **Top-k** (`40`) sets every token outside the 40 most likely to zero probability, which cuts off unlikely, nonsense tokens.
- Generation stops early if the model predicts `<|eof|>`.

In [ ]:
@torch.no_grad()
def generate(prompt, max_new=150, temperature=0.8, top_k=40):
    """Continue `prompt` one sampled token at a time."""
    model.eval()
    ids = torch.tensor([encode(prompt)], device=device)
    for _ in range(max_new):
        logits = model(ids[:, -SEQ:])[:, -1, :] / temperature   # last position only
        kth = torch.topk(logits, top_k).values[:, [-1]]          # score of the k-th best token
        logits[logits < kth] = -float("inf")                     # keep top-k
        nxt = torch.multinomial(F.softmax(logits, dim=-1), 1)    # pick one ID
        if nxt.item() == EOF_ID: break                           # model says "file over"
        ids = torch.cat([ids, nxt], dim=1)                       # append and continue
    model.train()
    return decode(ids[0].tolist())

# Try a few prompts, from a bare keyword to the start of a function
torch.manual_seed(42)
for prompt in ["def ", "class ", "    def __init__(self", "import os\n\ndef read_file(path):\n"]:
    print("=" * 60)
    print(generate(prompt))

**Output**

```code
============================================================
def line
    i+1204 = 28800 | data
        list = 5e + 5
    if new[1-99] == "0":
        return last(data, last)
    # This
    i = labels and [1:]
    for label in dataclass_pensions:
        last = None
        values.append(_mapping)
    add_from_data = []
    try:
        os = []
    except ValueError:
        return _un
============================================================
class <= protocols.Index(preferrame, ctx) and x or x == ':':
            return x.join(prefix)
        elif (code == x.isoptions:
            return sub.process_inner(x)
        else:
            b = type(x)
        if x < 1:
            raise ValueError("writec: %s" % (not host,))
        elif not inspect.is_compatibility():
============================================================
    def __init__(self, initial_of_safe_tail,
                 default_differing=None):
        self._signal_digits = _new_fork_event_release()
        self._current_diff = _ix_diff
        self._loop.clear()

        with self._dump():
            self.cv.clear()
        if key == _audit_loop:
            self._loop = self._emp_divateEventLine(cls
============================================================
import os

def read_file(path):

# --------------------

# CONDIAJCOREMADLER = False

class True(object):

    """RINT = [
    ALIG_KEYLICTION]
    __doc__ = os.path.join(basename)
    __path__ = path__,
    def __init__(self, pathname):
        self.__dype_base = path


class _PathLixMetaMixin(OptionGotocol
```

> 💡 **What to notice:** The output has *Python's shape* (indentation, `def`/`class`, `self.` attributes, `try/except`, `raise ValueError(...)`), but its meaning is nonsense and it would not run. That is what you would expect from a 0.9 M-parameter model trained for about a minute.

---
## ✅ Summary

| | |
|---|---|
| **Data** | Python 3.14 standard library: 423 files, 8.4 M characters |
| **Tokenizer** | Byte-level BPE, 512 tokens, ~2.15 characters per token |
| **Model** | 4-layer decoder-only transformer, width 128, 4 heads, RoPE, RMSNorm, GELU |
| **Parameters** | 918,656 |
| **Training** | 2,000 steps × 16 × 256 tokens ≈ 8.2 M tokens, ~70 s on `mps` |
| **Held-out loss** | 6.39 → **2.34** |

### 🚀 Where to go next
- **Scale up**: more width, layers and context.
- **Train longer**: the held-out loss was still falling at step 2,000.
- **Bigger vocabulary**: more merges mean fewer, more meaningful tokens per line.
- **More data**: add third-party packages for more varied code.
- **Modern extras**: a SwiGLU feed-forward, weight tying between the embedding and output tables, dropout, or a KV-cache for faster generation.